# Day 2: Cleaning, Feature Engineering, and Data Validation

Alright, Day 2! Yesterday we generated our raw data and spotted a few missing values in the `payment_method` column. 

Today's agenda:
1. **Clean the data:** Impute those missing values and ensure our data types are correct.
2. **Feature Engineering (RFM):** Calculate Recency, Frequency, and Monetary scores. This is the gold standard for retail customer analytics.
3. **Feature Engineering (Time-Series):** Create rolling statistics for our daily sales to help our forecasting models later.
4. **Data Validation:** Set up Great Expectations to automatically check our data quality.

Let's load up yesterday's raw data and get to work.

In [1]:
import pandas as pd
import numpy as np
import datetime
import great_expectations as gx
import warnings

# Keep the output clean
warnings.filterwarnings('ignore')

# 1. Load the raw data we saved yesterday
print("Loading raw data from Day 1...")
customers = pd.read_csv('data/customers_raw.csv', parse_dates=['join_date'])
products = pd.read_csv('data/products_raw.csv')
transactions = pd.read_csv('data/transactions_raw.csv', parse_dates=['date'])

print(f"Loaded {len(transactions)} transactions, {len(customers)} customers, and {len(products)} products.")

Loading raw data from Day 1...
Loaded 62611 transactions, 2000 customers, and 100 products.


### Step 1: Data Cleaning

First things first: handling those missing `payment_method` values. 

In a real business, we might investigate why they are missing (e.g., is a specific POS terminal broken?). For now, the safest statistical approach is to fill them with the most common payment method (the mode). We also need to make sure our numerical columns are actually numbers.

In [2]:
# Check the missing values again just to be sure
print("Missing values before cleaning:")
print(transactions.isnull().sum())
print("-" * 30)

# Impute missing payment methods with the mode (most frequent value)
mode_payment = transactions['payment_method'].mode()[0]
transactions['payment_method'] = transactions['payment_method'].fillna(mode_payment)

# Enforce correct data types (good habit to prevent downstream errors)
transactions['total_amount'] = transactions['total_amount'].astype(float)
transactions['quantity'] = transactions['quantity'].astype(int)

print("Data cleaned! Missing values handled.")

Missing values before cleaning:
transaction_id       0
date                 0
customer_id          0
product_id           0
quantity             0
total_amount         0
payment_method    1237
dtype: int64
------------------------------
Data cleaned! Missing values handled.


### Step 2: Feature Engineering - RFM Analysis

Now for the fun part. We need to summarize customer behavior. Instead of looking at thousands of individual transactions, we want to know:
- **Recency:** How many days since they last bought something? (Lower is better)
- **Frequency:** How many total transactions have they made? (Higher is better)
- **Monetary:** How much total money have they spent? (Higher is better)

Let's calculate this.

In [3]:
# We need a "current date" to calculate Recency. 
# Let's use the day after the last transaction in our dataset.
reference_date = transactions['date'].max() + datetime.timedelta(days=1)

# Group by customer and calculate the RFM metrics
rfm_data = transactions.groupby('customer_id').agg(
    recency=('date', lambda x: (reference_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('total_amount', 'sum')
).reset_index()

# Round monetary to 2 decimal places for cleanliness
rfm_data['monetary'] = rfm_data['monetary'].round(2)

# Merge these new features back into our main customers dataframe
customers = customers.merge(rfm_data, on='customer_id', how='left')

# If any customer has 0 transactions, fill with 0
customers[['recency', 'frequency', 'monetary']] = customers[['recency', 'frequency', 'monetary']].fillna(0)

print("RFM features calculated!")
display(customers[['customer_id', 'recency', 'frequency', 'monetary']].head())

RFM features calculated!


,customer_id,recency,frequency,monetary
0,CUST_00001,13,25,10987.70
1,CUST_00002,4,27,10848.37
2,CUST_00003,2,28,9846.20
3,CUST_00004,3,18,6966.94
4,CUST_00005,5,24,6900.81


### Step 3: Feature Engineering - Time-Series Rolling Stats

Later this week, we are going to build Prophet and LSTM models to forecast demand. Those models love context. Giving them a 7-day moving average helps them understand the short-term trend and smooth out daily noise (like a random Tuesday slump).

In [4]:
# Aggregate total sales by day
daily_sales = transactions.groupby('date')['total_amount'].sum().reset_index()
daily_sales = daily_sales.sort_values('date')

# Create rolling statistics (7-day window)
daily_sales['rolling_mean_7d'] = daily_sales['total_amount'].rolling(window=7).mean()
daily_sales['rolling_std_7d'] = daily_sales['total_amount'].rolling(window=7).std()

# Fill the first few NaN values created by the rolling window with the actual daily value
daily_sales['rolling_mean_7d'] = daily_sales['rolling_mean_7d'].fillna(daily_sales['total_amount'])
daily_sales['rolling_std_7d'] = daily_sales['rolling_std_7d'].fillna(0)

print("Time-series rolling features created!")
display(daily_sales.head())

Time-series rolling features created!


,date,total_amount,rolling_mean_7d,rolling_std_7d
0,2025-01-01,31199.16,31199.16,0.0
1,2025-01-02,42377.80,42377.80,0.0
2,2025-01-03,38068.35,38068.35,0.0
3,2025-01-04,49290.46,49290.46,0.0
4,2025-01-05,36620.54,36620.54,0.0


### Step 4: Data Validation with Great Expectations

This is where we separate the juniors from the seniors. If bad data enters our pipeline tomorrow (e.g., a negative transaction amount, or a duplicate ID), our models will output garbage. 

We are going to use **Great Expectations (GX)** to write "unit tests" for our data. If the data fails these tests, the pipeline should halt.

In [5]:
# Initialize an in-memory GX context 
context = gx.get_context()

# Load our cleaned transactions into a GX Validator
validator = context.sources.pandas_default.read_dataframe(transactions)

print("Running Data Quality Checks...")

# Rule 1: total_amount must be strictly greater than 0
validator.expect_column_values_to_be_between(
    column="total_amount", min_value=0.01, max_value=None
)

# Rule 2: quantity must be at least 1
validator.expect_column_values_to_be_between(
    column="quantity", min_value=1, max_value=None
)

# Rule 3: transaction_id must be completely unique
validator.expect_column_values_to_be_unique(column="transaction_id")

# Rule 4: No nulls in critical columns
validator.expect_column_values_to_not_be_null(column="date")
validator.expect_column_values_to_not_be_null(column="payment_method")

# Execute the validation
validation_results = validator.validate()

# Print a clean summary
print("\n--- Great Expectations Summary ---")
print(f"Overall Success: {validation_results.success}")
print(f"Tests Passed: {validation_results.statistics['successful_expectations']}")
print(f"Tests Failed: {validation_results.statistics['unsuccessful_expectations']}")

if validation_results.success:
    print("\nAll data quality checks passed! The dataset is production-ready.")
else:
    print("\nWARNING: Data quality checks failed! Check the logs.")

Running Data Quality Checks...


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]


--- Great Expectations Summary ---
Overall Success: True
Tests Passed: 5
Tests Failed: 0

All data quality checks passed! The dataset is production-ready.


### Wrap Up & Saving

Our data is clean, we've engineered some powerful features, and we've proven it passes our quality checks. 

Let's save these cleaned datasets to our `data/` folder so we can use them for clustering tomorrow.

In [7]:
# Save the cleaned datasets
customers.to_csv('data/customers_cleaned.csv', index=False)
transactions.to_csv('data/transactions_cleaned.csv', index=False)
daily_sales.to_csv('data/daily_sales_features.csv', index=False)

print("Cleaned and engineered data saved to the 'data/' directory.")
print("Day 2 Complete! See you tomorrow for Customer Segmentation.")

Cleaned and engineered data saved to the 'data/' directory.
Day 2 Complete! See you tomorrow for Customer Segmentation.
